# Beyond the Course: How MetaFor Works

This notebook explains everything in MetaFor that IS216 did not teach: the extra libraries, services, Vue features, browser APIs and database techniques. Each part starts from something you already know, then shows the real project code.

**How to read it**

| Label | Meaning |
|---|---|
| **Not in course** | No mention in the slides, speaker notes, PDFs or course code |
| **Seen, not explained** | Appears in starter code, a challenge or a passing mention, but no lesson teaches it |

**How the labels were decided:** every topic was searched for across the Week 1 to 11 slides and speaker notes, the challenge and extra-exercise PDFs, the lab test folders, and the `IS216` code repo.

**What that search could not cover:** before-class videos, the Google Docs linked from slides, and unopened zip files such as `Week9_InClass_V2.zip`. A topic labelled **Not in course** may still have come up in one of those.

## Contents

| Part | Topic |
|---|---|
| 1 | The full list at a glance |
| 2 | Nuxt UI and Tailwind CSS |
| 3 | Three.js |
| 4 | p5.js |
| 5 | Login: sessions, Spotify OAuth, encryption, origin checks |
| 6 | Spotify Web Playback SDK |
| 7 | Song mood: audio features, lyrics, DeepL, the emotion model |
| 8 | Groq, Last.fm and Open-Meteo |
| 9 | Vue features beyond the slides |
| 10 | Browser APIs |
| 11 | CSS beyond the slides |
| 12 | JavaScript patterns |
| 13 | MongoDB and Mongoose beyond the slides |
| 14 | Tooling, deployment and testing |
| 15 | Self-test questions |

## Part 1: The full list at a glance

**What the course does cover, so it is left out here:** HTML, CSS basics, Bootstrap, the DOM, `ref`, `computed`, directives, `onMounted`, `onUnmounted`, props, emits, named slots, `localStorage`, Axios methods, Vue Router with guards, Pinia, composables (optional), `watch` (in the Week 10 challenges), Vitest, Vue Test Utils, Playwright, Express, Mongoose schemas and `dotenv`.

### Libraries

| Library | Status | Used for |
|---|---|---|
| `@nuxt/ui` | Not in course | Ready-made buttons, inputs, alerts |
| `tailwindcss` | Seen, not explained (named once, in a "do not use" rule) | Utility classes and design tokens |
| `three` | Not in course | 3D objects and the shader background |
| `p5` | Not in course | Mood river and listening garden |
| `express-session` | Not in course | Login sessions |
| `connect-mongo` | Not in course | Stores sessions in MongoDB |
| `cors` | Seen, not explained (in starter servers) | Cross-origin rules |
| `@huggingface/transformers` | Not in course | Emotion model for lyrics |

### External services (all Not in course)

Spotify (login, Web API, Web Playback SDK), ReccoBeats, LRCLIB, DeepL, Last.fm, Groq, Open-Meteo.

### Vue and Vue Router

| Feature | Status |
|---|---|
| `<Teleport>`, `<Transition>` | Not in course |
| `defineAsyncComponent`, `defineModel`, `defineExpose`, `defineOptions` | Not in course |
| Scoped slots (`<slot :open="start">`) | Not in course (named slots are taught) |
| `onBeforeUnmount` | Not in course (`onUnmounted` is taught and is very similar) |
| `nextTick` | Seen, not explained (Week 10 test files only) |
| Lazy routes (`() => import(...)`) | Seen, not explained (scaffold code) |
| `v-show` | Seen, not explained (one speaker note) |
| `router.afterEach` | Seen, not explained (one line on a slide) |
| `scrollBehavior`, catch-all route | Not in course |
| `axios.create`, interceptors | Not in course |

### Browser APIs

| API | Status |
|---|---|
| `requestAnimationFrame`, `IntersectionObserver`, `ResizeObserver` | Not in course |
| Canvas 2D, WebGL, Web Audio | Not in course |
| Pointer events, `setPointerCapture` | Not in course |
| `AbortController`, `AbortSignal`, `navigator.share`, `<dialog>` | Not in course |
| `getBoundingClientRect`, `setProperty`, `getComputedStyle`, `visibilitychange` | Not in course |
| `matchMedia` | Seen, not explained (Bootstrap sample only) |
| `fetch` | Seen, not explained (named on one slide) |

### Database and tooling

| Topic | Status |
|---|---|
| Indexes, aggregation, `bulkWrite`, `$inc`, `$max`, `.lean()`, `select: false` | Not in course |
| `findOneAndUpdate` | Seen, not explained (blog server model) |
| Vite proxy, `node:test`, `page.route` | Not in course |
| Vercel | Seen, not explained (listed as a hosting option) |
| GitHub Actions | Seen, not explained (a Playwright installer option, set to false) |

## Part 2: Nuxt UI and Tailwind CSS

### Starting from Bootstrap

You already know the idea from Week 3: instead of writing CSS, you add ready-made classes. MetaFor splits that job in two:

- **Tailwind** gives small single-purpose classes. There is no `.card` class. You build the look from pieces like `flex`, `gap-3` and `mt-8`.
- **Nuxt UI** gives whole Vue components, such as `<UButton>`, which are styled with Tailwind.

| Bootstrap | Tailwind | Note |
|---|---|---|
| `d-flex` | `flex` | |
| `mt-3` | `mt-4` | Both are 1rem. Each Tailwind step is 0.25rem |
| `col-md-6` | `md:w-1/2` | `md:` means "from this screen width up" |
| `text-center` | `text-center` | |

### Reading a real class list

**From** [personality-quiz.vue](../client/src/views/personality/personality-quiz.vue)

```html
<main class="mx-auto w-[min(48rem,var(--page-width))] flex-1 pt-(--page-top) pb-(--page-bottom)">
```

- `mx-auto`: centres it, like `margin: auto` from Week 2.
- `w-[min(48rem,var(--page-width))]`: square brackets hold any custom CSS value.
- `pt-(--page-top)`: round brackets are shorthand for "use this CSS variable", so this means `padding-top: var(--page-top)`.

### Design tokens

Tailwind 4 is configured in CSS rather than a config file. Naming a colour under `@theme` creates matching classes, so `--color-brand-500` gives you `bg-brand-500` and `text-brand-500`.

**From** [main.css](../client/src/assets/css/main.css)

```css
@import 'tailwindcss';
@import '@nuxt/ui';
@theme static {
  --color-brand-500: #cd6651;
```

The word `static` keeps every variable in the final CSS, even if no class uses it. That matters here because `MoodRiver.vue` reads the mood colours from JavaScript, where Tailwind cannot see them being used.

### Nuxt UI setup takes three steps

1. **Register the plugin**, which makes `<UButton>` and friends available everywhere.

**From** [main.js](../client/src/main.js)

```js
import ui from '@nuxt/ui/vue-plugin'
app.use(ui)
```

2. **Add the Vite plugin** in [vite.config.js](../client/vite.config.js). This is where the custom colours are registered, which is what makes `<UBadge color="sad">` work.

3. **Wrap the app** in `<UApp>`, as [App.vue](../client/src/App.vue) does. Nuxt UI needs this root wrapper for its shared features, such as tooltips.

[app.config.ts](../client/app.config.ts) then restyles every button in one place: a `solid` button gets the project's `glass-action` class.

### Using a component

**From** [SongSearch.vue](../client/src/components/landing/SongSearch.vue)

```html
<UButton
  class="glass-action justify-center rounded-full font-sans"
  type="submit"
  size="lg"
  :loading="isAnalysing"
  :disabled="!chosen"
>
```

Everything here is props, as in Week 6. `:loading` adds a spinner without you writing one.

## Part 3: Three.js

**Used in:** [CassetteViewer.vue](../client/src/components/listening/CassetteViewer.vue), [LandingObject3D.vue](../client/src/components/landing/LandingObject3D.vue), [VinylTurntable.vue](../client/src/components/listening/VinylTurntable.vue), [WaveBackground.vue](../client/src/components/personality/WaveBackground.vue)

### The core idea

Three.js draws 3D using WebGL, which runs on the graphics card. Think of it as a film set:

| Three.js | Film set |
|---|---|
| Scene | The set, holding everything |
| Camera | Where you film from |
| Lights | Lighting |
| Mesh (geometry + material) | An actor: its shape plus its surface |
| Renderer | Turns all of it into pixels on a `<canvas>` |

### The cassette, step by step

**1. Set up the scene and camera.** The 32 is the field of view in degrees. A smaller number looks more zoomed in.

**From** [CassetteViewer.vue](../client/src/components/listening/CassetteViewer.vue)

```js
scene = new THREE.Scene()
camera = new THREE.PerspectiveCamera(32, 1, 0.1, 100)
camera.position.set(0, 0.1, 5.6)
```

**2. Add lights.** Without them, most materials render black.

```js
scene.add(new THREE.HemisphereLight(0xfff2df, 0x6d5261, 2.2))
```

**3. Add controls.** `OrbitControls` turns mouse drags into camera movement. The cassette stays still and the camera circles it.

**4. Load the model.** `.glb` is a 3D file format, and [cassette_tape.glb](../client/public/cassette_tape.glb) sits in `public/`. The code measures the model, centres it, and scales its longest side to 3.15 units, so the file's original size does not matter.

```js
model.position.sub(center)
model.scale.setScalar(3.15 / largestSide)
```

**5. Run the render loop.** A 3D scene is redrawn about 60 times a second. Skipping the draw while off-screen saves battery.

```js
function animate() {
  animationFrame = requestAnimationFrame(animate)
  if (!visible || document.hidden) return
  controls?.update()
  renderer?.render(scene, camera)
}
```

**6. Clean up.** The graphics card memory is not freed by the normal JavaScript clean-up, so `onBeforeUnmount` must call `dispose()` on each geometry, material and texture.

### Clicking a 3D object

A 3D object is not a DOM element, so `@click` cannot target it. [VinylTurntable.vue](../client/src/components/listening/VinylTurntable.vue) uses a **raycaster**, which shoots a line from the camera through the mouse position and reports what it hits.

Screen pixels are first converted to coordinates from -1 to 1, which is what Three.js expects. Note the minus sign on Y, because screen Y grows downward and 3D Y grows upward.

**From** [VinylTurntable.vue](../client/src/components/listening/VinylTurntable.vue)

```js
const ray = new THREE.Raycaster()
((event.clientX - bounds.left) / bounds.width) * 2 - 1,
(-(event.clientY - bounds.top) / bounds.height) * 2 + 1,
if (ray.intersectObjects(scene.children, true).some((hit) => hit.object.userData.playControl))
```

### Textures painted in code

[LandingObject3D.vue](../client/src/components/landing/LandingObject3D.vue) has no image files for the Spotify tile and journal. It paints textures on a hidden 2D canvas, then wraps them around the 3D shape.

**From** [LandingObject3D.vue](../client/src/components/landing/LandingObject3D.vue)

```js
const board = document.createElement('canvas')
const texture = new THREE.CanvasTexture(board)
```

It uses a **seeded** random function, so the "random" scratches come out the same on every visit.

### Shaders

[WaveBackground.vue](../client/src/components/personality/WaveBackground.vue) has no 3D objects at all. It uses a **shader**, a small program in the GLSL language that runs on the graphics card.

- The **fragment shader** runs once for every pixel, in parallel, and outputs that pixel's colour.
- **Uniforms** are values sent from JavaScript to the shader. Updating `uTime` each frame is what makes the waves move.

**From** [WaveBackground.vue](../client/src/components/personality/WaveBackground.vue)

```glsl
float wave1 = sin(p.x * 2.2 + t * 1.3 + sin(p.y * 1.7 - t) * 1.2);
vec3 colour = mix(lavender, terracotta, smoothstep(-1.0, 1.0, wave1) * 0.85);
```

`sin` makes a smooth wave, `smoothstep` turns it into a value from 0 to 1, and `mix` blends two colours by that value.

## Part 4: p5.js

**Used in:** [MoodRiver.vue](../client/src/components/dashboard/MoodRiver.vue), [ListeningArtwork.vue](../client/src/components/dashboard/ListeningArtwork.vue)

### What p5 gives you

p5 is a friendlier layer over the 2D canvas. You write two functions:

- `setup()` runs once.
- `draw()` runs repeatedly, once per frame.

Most p5 tutorials use global functions. That would clash in a Vue app, so MetaFor uses **instance mode**: all p5 functions hang off a `p` object and draw inside one chosen element.

p5 is also a heavy library, so it is imported only when needed (see lazy loading in Part 9).

**From** [MoodRiver.vue](../client/src/components/dashboard/MoodRiver.vue)

```js
const { default: P5 } = await import('p5')
instance = new P5((p) => {
  }, host.value)
```

### How the mood river is built

**Step 1: turn data into shapes.** Each day has play counts per mood. `buildShape` turns them into shares, then walks across the canvas every 3 pixels, working out each stream's top and bottom edges. It blends between neighbouring days so the river curves instead of stepping.

The blend uses a formula called smoothstep:

**From** [MoodRiver.vue](../client/src/components/dashboard/MoodRiver.vue)

```js
const smooth = (t) => t * t * (3 - 2 * t)
```

**Step 2: draw the streams.** Each stream is one closed shape: along its top edge left to right, then back along its bottom edge.

**Step 3: add moving particles.** Up to 650 small dots flow along the streams. A dot is more likely to appear where a stream is wide, and it moves faster when that mood had more plays that day.

To pick a position fairly, the code keeps running totals of stream width and uses a **binary search** on them. That is the same idea as "guess the middle, then discard half", which is much faster than checking each column.

**Step 4: pause when not needed.** `noLoop()` stops `draw()` from running and `loop()` restarts it. `redraw()` draws a single frame, which is how a hover still updates while paused.

**From** [MoodRiver.vue](../client/src/components/dashboard/MoodRiver.vue)

```js
if (paused.value || reduced.value || !inView.value || document.hidden) instance.noLoop()
else instance.loop()
```

### Connecting p5 and Vue

The two systems do not know about each other, so the component links them by hand:

- **Vue to p5:** `selectedIndex` is a `ref`, read inside `draw()`, and `instance?.redraw()` is called after it changes.
- **Colours:** read from your CSS with `getComputedStyle`, so the river matches the theme.
- **Accessibility:** a canvas is a blank image to a screen reader, so the code adds `role="img"` and an `aria-label`.

### The listening garden

[ListeningArtwork.vue](../client/src/components/dashboard/ListeningArtwork.vue) draws one flower per top artist, with more plays giving a bigger bloom. Three p5 functions do most of the work:

| Function | Purpose |
|---|---|
| `push()` / `pop()` | Save and restore the drawing position |
| `translate()` | Move where (0, 0) is |
| `rotate()` | Turn the drawing direction |

To draw petals, it moves to the flower's centre, then repeatedly rotates and draws the same petal shape. The petal code never needs to work out positions around a circle. It also reads live weather data, so wind makes the flowers sway more (see `sway` in the file).

### Three.js, p5 or plain canvas?

| Tool | Best for | Used for |
|---|---|---|
| Three.js | 3D objects, GPU effects | Cassette, turntable, waves |
| p5 | 2D generative art with many shapes | River, garden |
| Plain Canvas 2D | Simple or exact 2D drawing | Journal drawing, share card, margin notes |

## Part 5: Login: sessions, Spotify OAuth, encryption, origin checks

### Why not Week 11's method?

The Week 11 MiniShop solution stores a fake token in Pinia and `localStorage`. Any script running on the page can read `localStorage`, so a real login should not be kept there. MetaFor uses a **session** instead:

1. The server keeps the login record in MongoDB.
2. The browser only holds a random ID, in a cookie that JavaScript cannot read.

### `express-session` and `connect-mongo`

**From** [app.js](../server/app.js)

```js
    name: "metafor.sid",
    secret: process.env.SESSION_SECRET,
    resave: false,
    saveUninitialized: false,
    store: MongoStore.create({
      httpOnly: true,
      secure: process.env.NODE_ENV === "production",
      sameSite: "lax",
      maxAge: THIRTY_DAYS_MS,
```

| Option | What it does |
|---|---|
| `secret` | Signs the cookie, so a changed ID is rejected |
| `saveUninitialized: false` | No session is saved until there is something to store |
| `store` | `connect-mongo` keeps sessions in MongoDB, so they survive a restart |
| `httpOnly` | Page JavaScript cannot read the cookie |
| `secure` | The cookie is only sent over HTTPS in production |
| `sameSite: "lax"` | Other sites' background requests do not carry the cookie |

**Each request then works like this:** the browser sends the cookie automatically, `express-session` loads the session and sets `req.session`, then [requireUser.js](../server/middleware/requireUser.js) reads `req.session.userId` and loads the user.

This is also why [stores/auth.js](../client/src/stores/auth.js) calls `/api/me`. The page cannot read the cookie, so it has to ask the server who is logged in.

### Spotify login (OAuth)

OAuth lets users log in with Spotify without MetaFor ever seeing their Spotify password.

| Step | Where | What happens |
|---|---|---|
| 1 | Browser | Goes to `/api/auth/login` |
| 2 | Server | Creates a random `state`, saves it in the session, redirects to Spotify |
| 3 | Spotify | User approves the listed permissions (**scopes**) |
| 4 | Spotify | Redirects back to `/api/auth/callback?code=...&state=...` |
| 5 | Server | Checks `state` matches, which proves this login started here |
| 6 | Server | Swaps `code` plus the client secret for tokens, server to server |
| 7 | Server | Saves the user, starts a session, redirects into the app |

The code from [authController.js](../server/controller/authController.js) and [spotifyAuth.js](../server/services/spotifyAuth.js) has three security details worth explaining in a presentation:

- **`state`** stops another site from tricking your browser into finishing a login it started.
- **`req.session.regenerate`** swaps in a new session ID at login, so an ID set before login cannot be reused afterwards.
- **`safeReturnPath`** only accepts paths inside the app, so `?next=` cannot send people to another website.

### Two kinds of token

| Token | Lifetime | Stored | Used for |
|---|---|---|---|
| Access token | Short (the app reads `expires_in`) | Browser memory only | Calling Spotify |
| Refresh token | Long | MongoDB, encrypted | Getting new access tokens |

MetaFor also uses a third type, an **app token** (client credentials), for public song search. That is why logged-out visitors can search on the landing page.

### Encrypting the refresh token

A leaked refresh token would give access to someone's Spotify, so [crypto.js](../server/lib/crypto.js) encrypts it with AES-256-GCM before saving.

**From** [crypto.js](../server/lib/crypto.js)

```js
const iv = randomBytes(12);
const cipher = createCipheriv(ALGORITHM, key(), iv);
const tag = cipher.getAuthTag();
```

| Part | Purpose |
|---|---|
| Key (`TOKEN_KEY`) | The secret, 32 bytes, kept in `.env` |
| IV | Random per encryption, so equal text encrypts differently each time |
| Tag | A fingerprint. Decryption fails if anything changed |

Changing one byte of the encrypted data and then decrypting gives this error, which shows the tag doing its job:

```text
Unsupported state or unable to authenticate data
```

### `cors` and `apiSafety`

Browsers block a page from reading responses from another origin unless the server allows it, and `cors` sends that permission. The Week 5 starter server has it too, with this comment: `set to true only if you use cookies/sessions`.

MetaFor leaves `credentials` off because the browser only calls `/api` on its own origin. The Vite proxy and the Vercel rewrite forward those requests (see Part 14).

[apiSafety.js](../server/middleware/apiSafety.js) adds another layer. CORS only controls what a page may *read*, so a cross-site `POST` could still reach the server. This middleware rejects any write request whose `Origin` is not on the allowed list.

## Part 6: Spotify Web Playback SDK

**Used in:** [useListeningRoom.js](../client/src/composables/useListeningRoom.js), [spotify.js](../client/src/api/spotify.js)

An SDK is code a company provides for using its service. This one turns the browser tab into a Spotify device, like your phone appearing in Spotify's device list. It needs Spotify Premium.

### Loading a script at runtime

The SDK is not on npm. It loads from Spotify's site, so the code adds a `<script>` tag itself, and Spotify calls a global function once it is ready. Wrapping this in a Promise lets the rest of the code simply `await` it.

**From** [spotify.js](../client/src/api/spotify.js)

```js
window.onSpotifyWebPlaybackSDKReady = () => {
script.src = 'https://sdk.scdn.co/spotify-player.js'
```

### Creating the player

The SDK asks for a token through a callback whenever it needs one, and the browser gets that token from your server.

**From** [useListeningRoom.js](../client/src/composables/useListeningRoom.js)

```js
player ??= new Spotify.Player({
name: 'MetaFor Listening Room',
getOAuthToken: (cb) => {
```

### Events

The player reports changes through events, similar to `$emit` from Week 6, and the composable copies them into `ref`s so that Vue updates the page:

| Event | Meaning |
|---|---|
| `ready` | Gives a `device_id`, used to tell Spotify where to play |
| `player_state_changed` | Song, position or pause state changed |
| `account_error` | The account has no Premium |
| `autoplay_failed` | The browser blocked sound until the user interacts |

### Why the music keeps playing between pages

Component state is lost when you leave a page. Here, the `ref`s are created **outside** the exported function, at the top of the file. That means one shared copy exists for the whole app, and it survives route changes.

**From** [useListeningRoom.js](../client/src/composables/useListeningRoom.js)

```js
// Module singleton: playback survives Vue route changes.
const track = ref(emptyTrack())
```

This differs from the optional Week 11 `useCounter` composable, where each call gets its own state. `ListeningDock` also sits in `App.vue`, outside `<RouterView>`, so it never unmounts.

## Part 7: Song mood: audio features, lyrics, DeepL, the emotion model

**Used in:** [server/services/mood/](../server/services/mood/index.js)

### The pipeline

| Step | Source | Output |
|---|---|---|
| 1. Sound | ReccoBeats API | Valence, energy, danceability, acousticness (0 to 1) |
| 2. Lyrics | LRCLIB API | Plain lyrics text |
| 3. Translate | DeepL API, only if not English | English verses |
| 4. Emotion | Local model (Transformers.js) | Share of joy, sadness, anger and so on |
| 5. Decide | `mood.js` rules | A mood such as "Uplifting" |

If lyrics fail, the song still gets a sound-based reading. Lyric text is never saved. Only the scores and short highlights are kept, as `toStoredLyrics` in [analyse.js](../server/services/mood/analyse.js) shows.

### Running an AI model on your own server

A **Transformer** is a neural network that reads words in context. MetaFor runs a small one for free, with no outside API, through `@huggingface/transformers`.

**From** [emotion.js](../server/services/mood/emotion.js)

```js
import { env, pipeline } from '@huggingface/transformers'
const MODEL = 'onnx-community/emotion-english-distilroberta-base-ONNX'
classifier ??= pipeline('text-classification', MODEL, { dtype: 'q8' }).catch((err) => {
```

- **`pipeline`** handles the whole job: turning words into numbers, running the model, and producing labels.
- **`text-classification`** means "put this text into categories".
- **ONNX** is a portable model file format that can run in JavaScript.
- **`dtype: 'q8'`** uses a compressed version of the model, which is smaller and faster.
- **Downloading** happens once (about 83 MB, into `server/.models`). After that, it runs offline.

### From verses to one score

The model labels each verse with seven emotions, adding up to about 1. Each emotion is then given a positivity value. For example, joy is 0.95 and sadness is 0.1.

**Worked example:** a verse scores joy 0.6, sadness 0.2, neutral 0.2.

| Calculation | Result |
|---|---|
| Emotional part = 1 - neutral | 0.8 |
| Valence = (0.6 x 0.95 + 0.2 x 0.1) / 0.8 | 0.7375 |

Neutral is left out, so that mostly neutral verses do not drag everything toward the middle.

For the whole song, each verse's weight is its line count times how often it repeats, so a repeated chorus counts more. Verses with more emotion also count more.

The model has one known gap, which the code handles with a word list: it reads "outrun my gun" as only mild anger. `darkThemes` looks for words about guns, death or drugs, and needs at least two different ones, so a single "shooting star" does not trigger it.

### From scores to a mood

The rules in [mood.js](../server/services/mood/mood.js) are based on Russell's circumplex model, which places feelings on two axes:

|  | Low energy | High energy |
|---|---|---|
| **High valence** | Calm | Happy |
| **Low valence** | Sad | Sad |

A fifth check runs before this table: harsh, non-acoustic sound or angry lyrics over a heavy track can make a song **Tense** instead (see `isTense`).

Danceability then picks the specific name within that group. Lyrics can shift valence by at most 30% (`LYRIC_WEIGHT`).

**Worked example:** run through the real `readMoods` function.

| Input | Value |
|---|---|
| Audio valence, energy, danceability, acousticness | 0.8, 0.7, 0.5, 0.3 |
| Lyric valence, emotional share, joy | 0.25, 0.4, 0.2 |

```text
Lyric valence 0.25 is recentred to 0.5 (the model reads ordinary lyrics as about 0.25)
Lyric weight = 0.3 x min(1, 0.4 / 0.4) = 0.3
Blended valence = 0.7 x 0.8 + 0.3 x 0.5 = 0.71
Valence >= 0.5 and energy >= 0.5, so the group is Happy
Danceability 0.5 is under 0.6, so the mood is "Uplifting"
```

If sound and words clearly disagree, the clash becomes the mood. Bright music with dark lyrics becomes "Deceptively upbeat".

### The three APIs

| Service | Key needed | Worth noticing |
|---|---|---|
| [ReccoBeats](../server/services/mood/reccobeats.js) | No | Replaces Spotify's retired audio-features endpoint |
| [LRCLIB](../server/services/mood/lyrics.js) | No | Retries when busy, waiting as long as `Retry-After` says |
| [DeepL](../server/services/mood/translate.js) | Yes | Removes repeated verses first, so a chorus is only charged once |

### Background jobs

Analysis is slow, so it runs outside user requests. [jobs/sync.js](../server/jobs/sync.js) and [jobs/analyse.js](../server/jobs/analyse.js) are ordinary Node scripts, which GitHub Actions runs on a schedule (see Part 14).

## Part 8: Groq, Last.fm and Open-Meteo

### Groq: a hosted AI model

Groq runs large AI models and you call them over HTTP. MetaFor calls it in two places:

- [quizLlm.js](../server/services/quizLlm.js) writes the personality summary.
- [journalController.js](../server/controller/journalController.js) writes journal starters.

The API key stays on the server. The browser calls your Express route and Express calls Groq, because anything sent to the browser can be read by users.

An AI model's reply could be anything, so `quizLlm.js` controls it in layers:

| Layer | What it does |
|---|---|
| Limited choices | Scoring ranks the types first, and the model may only pick from the top 3 |
| JSON schema | `strict: true` forces the reply into an exact shape |
| Validation | `isValidAnalysis` checks the type and text lengths again |
| Timeout | Gives up after 15 seconds |
| Fallback | Any failure returns `null`, and the scoring result is used |

**From** [quizLlm.js](../server/services/quizLlm.js)

```js
typeId: { type: "string", enum: typeIds },
json_schema: { name: "personality_analysis", strict: true, schema: analysisSchema(typeIds) },
```

So the app still works with no Groq key at all. `temperature: 0.7` controls how varied the writing is, where lower values are more predictable.

The journal prompt has one more safeguard. Its system message says to treat the user's text "as a topic for reflection", which helps stop users from overriding the instructions.

### Last.fm: genres

Spotify no longer provides artist genres, so [artistGenres.js](../server/services/artistGenres.js) gets tags from Last.fm. Tags are written by listeners, so it filters out ones like "seen live" and "british". Results are cached for 7 days. After a failure, lookups pause for 10 minutes, so one outage does not slow down every quiz request.

### Open-Meteo: weather

[api/weather.js](../client/src/api/weather.js) calls Open-Meteo straight from the browser, unlike the others. This is safe because it needs no key. It sends `fetch` a timeout, and rejects replies with missing fields before using them. The code turns numeric weather codes into scenes for the garden, where wind makes the flowers sway.

## Part 9: Vue features beyond the slides

### `<Teleport>`: render somewhere else

A full-screen overlay nested deep in the page can get clipped by a parent's `overflow: hidden`. `<Teleport>` keeps the logic in your component but puts the HTML at the end of `<body>`.

**From** [StoryMode.vue](../client/src/components/landing/StoryMode.vue)

```html
<Teleport to="body">
```

### `<Transition>`: animate in and out

Normally, `v-if` removes an element at once, so CSS has no time to animate it. `<Transition>` delays the removal and adds classes at each stage. With `name="swap"`, you style `.swap-enter-active`, `.swap-leave-active`, `.swap-enter-from` and `.swap-leave-to`.

**From** [TasteQuiz.vue](../client/src/components/landing/TasteQuiz.vue)

```html
<Transition name="swap" mode="out-in">
  <div v-if="!done" :key="step" class="taste-question">
```

- `mode="out-in"` waits for the old content to leave before the new content enters.
- `:key="step"` matters because each question uses the same `<div>`, and Vue would normally just update the text. A new key tells Vue it is a new element, which triggers the animation.

### Lazy loading

Three.js is large. If everything loaded up front, every visitor would wait for it. `import()` as a function downloads a file only when it is needed, and Vite splits it into its own file automatically.

| Form | Where |
|---|---|
| `() => import('../views/landing.vue')` | Routes, in the course scaffold too |
| `defineAsyncComponent(() => import(...))` | Components, as in `landing.vue` |
| `await import('p5')` | Any library, as in `MoodRiver.vue` |

### `defineModel`: `v-model` on your own component

From Week 6, you would need a prop plus an emit. `defineModel` does both in one line, and assigning to it notifies the parent.

**From** [JournalSongPicker.vue](../client/src/components/journal/JournalSongPicker.vue)

```js
const song = defineModel({ type: Object, default: null })
```

### `defineExpose`: let a parent call a method

Everything in `<script setup>` is private by default. `defineExpose` opens up chosen items. Here, a parent holding a `ref` to this component can call its `open()`.

**From** [ShareCardDialog.vue](../client/src/components/personality/ShareCardDialog.vue)

```js
defineExpose({ open })
```

### Scoped slots: a child passes data to the slot

You learned that slots let a parent fill in a child's content. In a **scoped** slot, the child also hands data back. `StoryMode` lets each parent design its own button, while giving it the `open` function to call.

**From** [StoryMode.vue](../client/src/components/landing/StoryMode.vue)

```html
<slot name="trigger" :open="start">
```

**From** [navbar.vue](../client/src/components/layout/navbar.vue)

```html
<template #trigger="{ open }">
```

### `nextTick`: wait for the DOM to update

Changing a `ref` does not update the page immediately. Vue batches changes and applies them shortly after. If you need the new element straight away, you have to wait first. `textInput` below does not exist in the DOM until Vue renders it.

**From** [JournalCanvas.vue](../client/src/components/journal/JournalCanvas.vue)

```js
nextTick(() => textInput.value?.focus())
```

### Template refs

`ref="canvas"` on an element fills a matching `const canvas = ref(null)` once mounted, and this gives direct access to the DOM element. Libraries such as Three.js need the real element. On a component instead of an element, you get the component, and `.$el` gets its root element, as in [MusicJournal.vue](../client/src/components/landing/MusicJournal.vue).

### Smaller items

| Feature | What it does | Example |
|---|---|---|
| `onBeforeUnmount` | Runs just before removal, while the DOM still exists | Most animated components |
| `defineOptions` | Sets component options such as `name` | `landing.vue` |
| `v-show` | Hides with CSS and keeps the element alive | `MusicJournal.vue` keeps its 3D canvases ready |
| `watch` with a getter | Watches part of an object | `App.vue` watches `auth.user?.id` |
| `{ immediate: true }` | Also runs the watcher once at the start | `App.vue` |
| `:key="$route.path"` | Forces a fresh page component on each path | `App.vue` |

### Vue Router extras

From [router/index.js](../client/src/router/index.js):

| Feature | What it does |
|---|---|
| `/:pathMatch(.*)*` | Catch-all, so unknown URLs redirect to `/` |
| `scrollBehavior` | Restores your scroll position on Back, otherwise goes to the top |
| `afterEach` | Sets the page title after each navigation |
| `meta: { title }` | Extra per-route data (Week 11 used `requiresAuth`) |

### Axios instance and interceptors

Week 5 called `axios.get()` directly, so every call repeated its setup and error handling. MetaFor creates one configured instance in [http.js](../client/src/api/http.js). An **interceptor** runs on every response before your code sees it:

**From** [http.js](../client/src/api/http.js)

```js
export const api = axios.create({
api.interceptors.response.use(
  (response) => response.data,
```

| Without interceptor | With interceptor |
|---|---|
| `(await axios.get(url)).data` | `await api.get(url)` |
| Each page reads `error.response.data.error` | `error.message` is always readable |

## Part 10: Browser APIs

### `requestAnimationFrame`: smooth animation

`setInterval` runs on its own schedule, so it can fall out of step with the screen. `requestAnimationFrame` runs just before the next screen repaint, usually 60 times a second, and pauses in background tabs. The function schedules itself again each time.

Screen speeds differ, and a 120 Hz screen calls it twice as often. So [StoryMode.vue](../client/src/components/landing/StoryMode.vue) measures real elapsed time rather than counting frames:

**From** [StoryMode.vue](../client/src/components/landing/StoryMode.vue)

```js
frame = requestAnimationFrame(tick)
if (!paused.value) elapsed.value += time - lastTime
```

### `IntersectionObserver`: is it on screen?

The old way is to check position on every scroll event, which is slow. An observer lets the browser tell you. `threshold: 0.4` means "tell me when 40% is visible".

**From** [ShowcasePreview.vue](../client/src/components/landing/ShowcasePreview.vue)

```js
observer = new IntersectionObserver(
if (!entry.isIntersecting) return
{ threshold: 0.4 },
```

### `ResizeObserver`: did this element change size?

CSS `@media` only knows about the window, but this watches one element. A canvas needs it because its drawing size has to be updated by hand.

### Canvas 2D

A canvas is a grid of pixels you draw on with commands. Once drawn, shapes are just pixels. There are no elements to click or style, unlike SVG or HTML.

```js
const ctx = canvas.getContext('2d')
ctx.fillStyle = 'red'
ctx.fillRect(10, 10, 100, 50)
```

**Sharp drawing on high-resolution screens.** Phones may have 2 or 3 real pixels per CSS pixel. The fix is to make the canvas bigger, then scale the coordinates back.

**From** [NoteMargins.vue](../client/src/components/landing/NoteMargins.vue)

```js
const ratio = window.devicePixelRatio || 1
canvas.value.width = width * ratio
context.setTransform(ratio, 0, 0, ratio, 0, 0)
```

**Where the project uses it:**

| File | Purpose |
|---|---|
| [JournalCanvas.vue](../client/src/components/journal/JournalCanvas.vue) | Drawing in the journal |
| [NoteMargins.vue](../client/src/components/landing/NoteMargins.vue) | Floating margin notes |
| [shareCard.js](../client/src/utils/shareCard.js) | Generates a 1080 x 1920 share image |

The share card needs two extra steps. Fonts have to finish loading first (`document.fonts.load`), and `canvas.toBlob` then turns the drawing into a PNG file.

### Web Audio API: making sound with no audio file

[NoteMargins.vue](../client/src/components/landing/NoteMargins.vue) builds a sound by connecting nodes, like plugging in guitar pedals: oscillator, then gain (volume), then speakers.

**From** [NoteMargins.vue](../client/src/components/landing/NoteMargins.vue)

```js
const oscillator = audio.createOscillator()
gain.gain.exponentialRampToValueAtTime(0.0001, now + 1.1)
oscillator.connect(partial).connect(gain)
```

- An **oscillator** produces a pure tone at a chosen frequency.
- The volume ramp is what makes it sound like a bell instead of a beep.
- Browsers block sound until the user clicks, so audio starts on the first click.
- Notes use a pentatonic scale, which sounds pleasant in any combination.

### Pointer events: one API for mouse, touch and pen

`pointerdown`, `pointermove` and `pointerup` work for all input types. During a drag, a fast mouse can leave the element and the moves would be lost. `setPointerCapture` fixes this by sending all pointer events to that element until release.

**A drag in three parts**, from [MusicJournal.vue](../client/src/components/landing/MusicJournal.vue):

```js
start = { x: event.clientX - offset.value.x, y: event.clientY - offset.value.y }
tile.value.setPointerCapture(event.pointerId)
offset.value = { x: event.clientX - start.x, y: event.clientY - start.y }
```

The `offset` is bound to `transform: translate(...)`. On release, `getBoundingClientRect()` gives on-screen positions, which are used to test whether the tile landed on the journal.

### `matchMedia`: asking about user settings

Some people get motion sickness and turn on "reduce motion" in their system settings. MetaFor checks this throughout.

**From** [landing.vue](../client/src/views/landing.vue)

```js
if (window.matchMedia('(prefers-reduced-motion: reduce)').matches) {
```

### `fetch` compared with Axios

The project uses Axios for its own API, and `fetch` for outside services.

| | Axios | `fetch` |
|---|---|---|
| Installed | Library | Built in |
| 404 or 500 | Throws an error | Does **not** throw, so check `response.ok` |
| JSON | Automatic | `await response.json()` |
| Timeout | `timeout` option | `AbortSignal.timeout(ms)` |

### `AbortController`: cancel a request

When the user searches again, the old request is no longer needed. Aborting it saves work, and stops an old reply from arriving late.

**From** [useListeningRoom.js](../client/src/composables/useListeningRoom.js)

```js
searchController?.abort()
searchController = new AbortController()
const found = await searchSpotify(query.value.trim(), searchController.signal)
```

An aborted request throws an `AbortError`, which the code deliberately does not show to the user.

### Smaller items

| API | What it does | Used in |
|---|---|---|
| `<dialog>`, `showModal()` | Built-in modal with focus handling | `ShareCardDialog.vue` |
| `navigator.share` | Opens the phone's share sheet | `ShareCardDialog.vue` |
| `navigator.canShare` | Checks if files can be shared, otherwise it downloads | `ShareCardDialog.vue` |
| `URL.createObjectURL` | Makes a temporary URL for a file in memory | `ShareCardDialog.vue` |
| `visibilitychange` | Fires when the tab is hidden or shown | `MoodRiver.vue` |
| `getComputedStyle` | Reads final CSS values from JavaScript | `MoodRiver.vue` |
| `style.setProperty` | Sets a CSS variable from JavaScript | `landing.vue` |
| `element.getBoundingClientRect` | Gives on-screen position and size | Many components |

## Part 11: CSS beyond the slides

### CSS variables set from JavaScript

The Bootstrap samples and scaffold CSS use `var(--x)`, but the slides do not explain it. MetaFor uses them to connect JavaScript and CSS: JavaScript provides the number, and CSS decides how it looks. This avoids re-rendering Vue on every mouse move.

**From** [landing.vue](../client/src/views/landing.vue)

```js
card.style.setProperty('--ry', `${(x - 0.5) * 2 * MAX_TILT}deg`)
card.style.setProperty('--rx', `${(0.5 - y) * 2 * MAX_TILT}deg`)
```

Step by step, for a card:

1. `x` runs from 0 (left edge) to 1 (right edge).
2. `x - 0.5` runs from -0.5 to 0.5, so the centre is 0.
3. Times 2 gives -1 to 1, and times `MAX_TILT` (5) gives -5 to 5 degrees.

### `color-mix()`

Blends two colours without any JavaScript. This mixes 22% of the chosen mood colour with transparent, which makes a soft glow.

**From** [landing.vue](../client/src/views/landing.vue)

```css
color-mix(in srgb, var(--mood-colour, transparent) 22%, transparent),
```

The second value in `var(--mood-colour, transparent)` is a fallback for when the variable is not set.

### Other items

| Feature | What it does |
|---|---|
| `radial-gradient` | A colour fade that spreads out from a point |
| `::before` with `content: ''` | Adds a decorative layer with no extra HTML |
| `isolation: isolate` | Keeps `z-index: -1` layers from slipping behind the page's background |
| `inset: 0 0 auto` | Shorthand for `top`, `right`, `bottom`, `left` |
| `@keyframes` | Named animations. Seen in the Week 2 and 4 in-class files, not explained in slides |

## Part 12: JavaScript patterns

### Debouncing: wait until typing stops

Sending a request per key press is wasteful. Instead, restart a timer on each key press, so the search only runs after 300 ms of silence.

**From** [SongSearch.vue](../client/src/components/landing/SongSearch.vue)

```js
window.clearTimeout(searchTimer)
searchTimer = window.setTimeout(() => findSuggestions(text.trim()), 300)
```

### Ignoring stale responses

Network replies can arrive out of order. If you type "ad" then "adele", the "ad" reply might come last and overwrite the correct results. Each search takes a number, and only the newest one may update the page.

**From** [SongSearch.vue](../client/src/components/landing/SongSearch.vue)

```js
const search = ++latestSearch
const songs = await searchSongs(text)
if (search === latestSearch) {
```

The same idea appears with `connectVersion` in `useListeningRoom.js` and `revision` in `stores/auth.js`.

### Clean up everything you start

Leaving a page removes the component, but timers, listeners and animation loops keep running unless you stop them. That wastes memory, and can crash when they touch elements that are gone. Almost every animated component follows this pairing:

| Start | Stop |
|---|---|
| `setInterval` | `clearInterval` |
| `setTimeout` | `clearTimeout` |
| `addEventListener` | `removeEventListener` |
| `requestAnimationFrame` | `cancelAnimationFrame` |
| `new ResizeObserver` | `.disconnect()` |

### Sharing one Promise

If three components ask "who is logged in?" at once, only one request should go out. So the code stores the Promise, and everyone awaits the same one.

**From** [stores/auth.js](../client/src/stores/auth.js)

```js
pending ??= getMe()
```

`a ??= b` means "if `a` is null or undefined, set it to `b`". The emotion model loads the same way.

### Caching in a `Map`

[spotifyAuth.js](../server/services/spotifyAuth.js) keeps recent searches in a `Map` for 10 minutes, up to 200 entries. A `Map` remembers insertion order, so the first key is the oldest one, and that is the one removed when the cache is full.

**From** [spotifyAuth.js](../server/services/spotifyAuth.js)

```js
searchCache.delete(searchCache.keys().next().value);
```

### Respecting rate limits

When Spotify answers `429 Too Many Requests`, it includes a `Retry-After` header. The code then stops calling until that time has passed, instead of making things worse.

### A timeout with `Promise.race`

`Promise.race` settles with whichever Promise finishes first. Racing a slow task against a timer gives a timeout for something that does not support one, as `connectPlayer` does in [useListeningRoom.js](../client/src/composables/useListeningRoom.js).

## Part 13: MongoDB and Mongoose beyond the slides

### Indexes

Without an index, MongoDB checks every document, like reading a whole book to find one word. An index is the list at the back of the book.

**From** [Listen.js](../server/models/Listen.js)

```js
listenSchema.index({ userId: 1, playedAt: -1 }, { unique: true });
```

- `1` means ascending and `-1` means descending, so this finds one user's newest plays quickly.
- `unique: true` also protects the data. The same play cannot be saved twice, even if two sync jobs overlap.

[config/db.js](../server/config/db.js) sets `autoIndex: false`, so indexes are created by one script ([db-init.js](../server/scripts/db-init.js)) instead of on every start.

### Aggregation pipelines

A pipeline passes documents through stages, like an assembly line. This one gets each quiz's latest attempt.

**From** [QuizAttempt.js](../server/models/QuizAttempt.js)

```js
{ $match: { userId } },
{ $sort: { createdAt: -1 } },
{ $group: { _id: "$quizId", attempt: { $first: "$$ROOT" } } },
{ $replaceRoot: { newRoot: "$attempt" } },
```

| Stage | What it does |
|---|---|
| `$match` | Keeps only this user's attempts |
| `$sort` | Newest first |
| `$group` | One bucket per quiz, keeping the first (so newest) document. `$$ROOT` means the whole document |
| `$replaceRoot` | Unwraps it, back to a normal attempt |

### Upsert: update, or create if missing

**From** [QuizAttempt.js](../server/models/QuizAttempt.js)

```js
return Personality.findOneAndUpdate({ userId }, { ...personality, userId }, {
upsert: true,
runValidators: true,
```

`runValidators` is needed because Mongoose skips schema validation on updates by default. `authController.js` uses the same upsert for login, so the first login creates the account.

### `bulkWrite`: many operations in one trip

Syncing can touch 50 songs. One request per song would be slow, so `bulkWrite` sends them together. `ordered: false` means one failure (such as a duplicate) does not stop the rest.

**From** [analysis.js](../server/services/analysis.js)

```js
await Track.bulkWrite(
return { updateOne: { filter: { _id: id }, update } };
{ ordered: false },
```

### Update operators

| Operator | Meaning | Why here |
|---|---|---|
| `$set` | Set these fields, leaving others alone | Most updates |
| `$inc` | Add to a number | Counting retry attempts |
| `$max` | Only update if the new value is larger | A sync cursor that can never go backwards |
| `$setOnInsert` | Only set when creating | Song data that should not be overwritten |
| `$unset` | Remove a field | Clearing the last error |

These are also safe when two jobs run at once, because the database applies them as one step. Reading a value, changing it in JavaScript and saving it could lose an update.

### Query helpers

| Feature | What it does | Example |
|---|---|---|
| `select: false` | Field is left out of queries by default | `refreshToken` in `User.js` |
| `.select("+field")` | Asks for that hidden field | `spotifyController.js` |
| `.lean()` | Returns plain objects, which is faster for read-only data | `jobs/sync.js` |
| `timestamps` | Adds `createdAt` and `updatedAt` | `QuizAttempt.js` |

`select: false` means that even a careless query cannot leak the token to the browser.

## Part 14: Tooling, deployment and testing

### The Vite proxy

The client (Vite) and API (Express) run on different ports, so they count as different origins. The proxy makes `/api` requests go through Vite to Express, so the browser only sees one origin and the session cookie just works.

**From** [vite.config.js](../client/vite.config.js)

```js
'/api': {
target: env.API_PROXY_TARGET || 'http://localhost:3000',
```

Spotify only accepts `127.0.0.1` redirects, which is why the dev server uses that address instead of `localhost`. They count as different sites for cookies.

### Vercel

[vercel.json](../vercel.json) does the same job in production. It deploys two services from one repo and routes by path:

| Path | Goes to |
|---|---|
| `/api/...` | The Express service |
| Anything else | The Vue client |

The client's own rewrite sends every path to `index.html`. That is needed for a single-page app: reloading `/dashboard` has no real file behind it, so Vue Router has to handle it.

### GitHub Actions

Actions runs scripts on GitHub's computers when something happens. MetaFor has two workflows.

| Workflow | Runs when | What it does |
|---|---|---|
| [e2e.yml](../.github/workflows/e2e.yml) | Pushes and pull requests | Runs the Playwright tests |
| [sync.yml](../.github/workflows/sync.yml) | On a schedule | Saves new plays, analyses new songs |

**From** [sync.yml](../.github/workflows/sync.yml)

```yaml
- cron: "7,22,37,52 * * * *"
```

A **cron** expression has five fields: minute, hour, day, month, weekday. This one means "at minutes 7, 22, 37 and 52 of every hour". Spotify only keeps a user's latest 50 plays, so history is lost unless it is saved often.

Secrets such as `MONGODB_URI` are stored in the repo settings, rather than in the code. The 83 MB model is kept between runs using `actions/cache`.

### Testing techniques beyond Week 10

**Node's own test runner.** The server uses `node:test`. It works like Vitest (`describe`, `it`) but needs no install, and `assert` replaces `expect`.

**From** [analyse.test.js](../server/tests/analyse.test.js)

```js
import assert from "node:assert/strict";
import { describe, it } from "node:test";
```

**Dependency injection.** How do you test code that calls four outside services? `analyseSong` takes those services as a parameter, with the real ones as the default. Tests pass in fake versions, so no network or model download is needed. This has the same purpose as stubs in Week 10, but for plain functions.

**From** [analyse.js](../server/services/mood/analyse.js)

```js
export async function analyseSong(track, deps = defaultDeps) {
```

**Fake network in Playwright.** Week 10's tests use the real app. [helpers.js](../client/e2e/helpers.js) intercepts requests, so tests never reach the real database and each test owns its data.

**From** [helpers.js](../client/e2e/helpers.js)

```js
await page.route(/^https:\/\//, (route) => route.abort())
const reply = (json, status = 200) => route.fulfill({ status, json })
```

**Fake network in Axios.** [http.spec.js](../client/src/__tests__/http.spec.js) replaces Axios's `adapter`, the part that actually sends requests, with a function that answers directly.

## Part 15: Self-test questions

Useful for the Week 13 Q&A. Try to answer before opening each one.

<details><summary>1. Why does the page call /api/me instead of reading the login cookie?</summary>

The cookie is `httpOnly`, so page JavaScript cannot read it. That protects it from malicious scripts. Only the server can tell who it belongs to.
</details>

<details><summary>2. What does `state` protect against in the Spotify login?</summary>

It proves the callback belongs to a login that started in this browser, so another site cannot make your browser finish a login it started.
</details>

<details><summary>3. Why can't the Groq key be in the Vue code?</summary>

All client code is sent to the browser, so anyone could read the key. The browser calls Express, and only Express calls Groq.
</details>

<details><summary>4. What happens if Groq is down during a quiz?</summary>

`analysePersonality` returns `null` and the app uses the scoring result, so the quiz still works.
</details>

<details><summary>5. Why does a Three.js component need `dispose()` when Vue already removes the component?</summary>

Geometries, materials and textures are stored in graphics card memory, which JavaScript's clean-up does not free. Without `dispose()`, memory use grows on each visit.
</details>

<details><summary>6. Why `v-show` in MusicJournal rather than `v-if`?</summary>

`v-if` would destroy the 3D canvases, and they would have to rebuild when pulled apart. `v-show` only hides them, so it is instant.
</details>

<details><summary>7. Why does Transition need `:key` in TasteQuiz?</summary>

Each question reuses the same `<div>`, so Vue would just change the text. A new key makes Vue treat it as a new element, which triggers the animation.
</details>

<details><summary>8. What are debouncing and the stale-response check each solving?</summary>

Debouncing reduces how many requests are sent. The stale check handles replies arriving out of order. They solve different problems, so `SongSearch` uses both.
</details>

<details><summary>9. Why does `fetch` code check `response.ok` while Axios code doesn't?</summary>

`fetch` only throws on network failure, so a 404 or 500 counts as success. Axios throws for those statuses automatically.
</details>

<details><summary>10. Why use `$max` for the sync cursor?</summary>

If two sync runs overlap, an older run finishing late would otherwise move the cursor backward. `$max` only ever moves it forward.
</details>

<details><summary>11. Why is lyric text not saved?</summary>

Only scores and short excerpts are kept, as `toStoredLyrics` shows. The project's README states that lyric text is not saved in the database.
</details>

<details><summary>12. Why does the server use a word list if it has an AI model?</summary>

The emotion model reads the feeling of the words, not the subject, so it scores "outrun my gun" as only mild anger. The word list catches dark subjects it misses.
</details>